# V15: 하이브리드 전략 (앙상블 예측 + 사후 수동 보정)

v14의 성능 하락 원인을 분석하여, 모델이 학습하기 어려운 미세 변동 요인(날씨, 공휴일 등)을 '사후 보정(Weak Correction)' 레이어로 분리했습니다.

## 주요 전략
1. **베이스 모델 고도화**: v13의 XGB, LGBM, CatBoost 트리플 앙상블 및 5-Fold CV 유지
2. **피처 최적화**: 모델이 주요 운영 지표에 집중하도록 날씨/공휴일 피처를 모델 입력에서 제외
3. **사후 보정 레이어**: 팀원 모델에서 점수 향상이 검증된 'Weather/Holiday/Menu Weak Correction' 로직 적용

In [1]:
import os
import re
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

In [2]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
weather_train = pd.read_csv('data/weather.csv')
weather_test = pd.read_csv('data/weatherTest.csv')

# 날씨 데이터 통합
weather = pd.concat([weather_train, weather_test], axis=0)
weather['일시'] = pd.to_datetime(weather['일시'])
weather['기온'] = pd.to_numeric(weather['기온'], errors='coerce').interpolate().bfill().ffill()
weather['강수량'] = pd.to_numeric(weather['강수량'], errors='coerce').fillna(0)
weather['is_rain'] = (weather['강수량'] > 0).astype(int)
weather['is_hot'] = (weather['기온'] >= 28).astype(int)
weather['is_cold'] = (weather['기온'] <= 5).astype(int)
weather = weather[['일시', '기온', '강수량', 'is_rain', 'is_hot', 'is_cold']]

In [3]:
# 2. 피처 엔지니어링 (보정용 피처와 모델용 피처를 함께 생성)
def normalize_menu_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('/', ' ').replace('&', ' ').replace('+', ' ')
    text = text.replace('(', ' ').replace(')', ' ')
    text = re.sub(r'[^가-힣A-Za-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

lunch_keyword_map = {
    'chicken': '치킨', 'donkatsu': '돈까스', 'jeyuk': '제육', 'curry': '카레',
    'fried_rice': '볶음밥', 'bibimbap': '비빔밥', 'jjajang': '짜장', 'tangsuyuk': '탕수육'
}

def get_lunch_keywords(text):
    text = normalize_menu_text(text)
    return {f'kw_{k}': int(v in text) for k, v in lunch_keyword_map.items()}

def get_dinner_style(text):
    text = normalize_menu_text(text)
    flags = {'style_chinese': 0, 'style_western': 0, 'style_japanese': 0, 'style_snack': 0, 'style_fusion': 0}
    chinese_kw = ['짜장', '짬뽕', '탕수', '마파', '깐풍', '유산슬']
    western_kw = ['돈까스', '스테이크', '파스타', '스프', '햄버거', '피자', '리조또']
    japanese_kw = ['초밥', '우동', '가라아게', '돈부리', '소바']
    snack_kw = ['떡볶이', '순대', '튀김', '라볶이', '김밥', '분식', '어묵']
    fusion_kw = ['브리또', '타코', '샐러드', '또띠아', '퓨전']

    if any(k in text for k in chinese_kw): flags['style_chinese'] = 1
    if any(k in text for k in western_kw): flags['style_western'] = 1
    if any(k in text for k in japanese_kw): flags['style_japanese'] = 1
    if any(k in text for k in snack_kw): flags['style_snack'] = 1
    if any(k in text for k in fusion_kw): flags['style_fusion'] = 1
    return flags

def add_features(df):
    df['일자'] = pd.to_datetime(df['일자'])
    df = df.sort_values('일자').reset_index(drop=True)
    df = pd.merge(df, weather, left_on='일자', right_on='일시', how='left')
    
    df['월'] = df['일자'].dt.month
    df['일'] = df['일자'].dt.day
    df['요일'] = df['일자'].dt.weekday
    
    # 식사가능자수
    df['식사가능자수'] = (df['본사정원수'] - df['본사휴가자수'] - df['본사출장자수'] - df['현본사소속재택근무자수']).clip(lower=1)
    
    # 운영 피처 (팀원 모델 기반)
    month_end = df['일자'] + pd.offsets.MonthEnd(0)
    df['days_to_month_end'] = (month_end - df['일자']).dt.days
    df['is_month_end_part'] = (df['days_to_month_end'] <= 5).astype(int)
    df['is_wed'] = (df['요일'] == 2).astype(int)
    df['is_fri'] = (df['요일'] == 4).astype(int)
    df['has_overtime'] = (df['본사시간외근무명령서승인건수'] > 0).astype(int)
    df['log_overtime'] = np.log1p(df['본사시간외근무명령서승인건수'])
    
    # 공휴일 변수 (보정용)
    df['holiday_after'] = (df['일자'].diff().dt.days >= 2).astype(int)
    df['holiday_before'] = (df['일자'].shift(-1).sub(df['일자']).dt.days >= 2).astype(int)
    
    # 메뉴 피처 (보정용)
    lunch_menu_flags = pd.DataFrame([get_lunch_keywords(x) for x in df['중식메뉴'].fillna("")])
    dinner_menu_flags = pd.DataFrame([get_dinner_style(x) for x in df['석식메뉴'].fillna("")])
    df = pd.concat([df, lunch_menu_flags, dinner_menu_flags], axis=1)
    
    return df

In [4]:
train_df = add_features(train)
test_df = add_features(test)

# 타겟 참여율
train_df['중식참여율'] = (train_df['중식계'] / train_df['식사가능자수']).clip(lower=0, upper=1.5)
train_df['석식참여율'] = (train_df['석식계'] / train_df['식사가능자수']).clip(lower=0, upper=1.5)

# 모델 학습용 특징 (날씨, 공휴일, 메뉴 제외)
lunch_train_features = [
    '월','일','요일','식사가능자수','본사출장자수','본사시간외근무명령서승인건수',
    'is_fri','days_to_month_end','is_month_end_part'
]
dinner_train_features = [
    '월','일','요일','식사가능자수','본사출장자수','본사시간외근무명령서승인건수',
    'is_wed','has_overtime','log_overtime'
]

In [5]:
# 3. 트리플 앙상블 학습 함수 (v13 개선)
def train_ensemble(train_df, features, target_col, test_df):
    X = train_df[features]
    y = train_df[target_col]
    X_test = test_df[features]
    
    num_folds = 5
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    
    test_preds = np.zeros(len(X_test))
    train_preds = np.zeros(len(X))
    
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        params = {
            'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 5, 'random_state': SEED
        }
        
        m_xgb = XGBRegressor(objective='reg:squarederror', **params)
        m_lgbm = LGBMRegressor(verbose=-1, **params)
        m_cat = CatBoostRegressor(verbose=0, **params)
        
        m_xgb.fit(X_tr, y_tr)
        m_lgbm.fit(X_tr, y_tr)
        m_cat.fit(X_tr, y_tr)
        
        split_test_pred = (m_xgb.predict(X_test) + m_lgbm.predict(X_test) + m_cat.predict(X_test)) / 3
        split_val_pred = (m_xgb.predict(X_val) + m_lgbm.predict(X_val) + m_cat.predict(X_val)) / 3
        
        test_preds += split_test_pred / num_folds
        train_preds[val_idx] = split_val_pred
        
    return train_preds, test_preds

print("Training Lunch...")
lunch_train_res, lunch_test_base = train_ensemble(train_df, lunch_train_features, '중식참여율', test_df)
print("Training Dinner...")
dinner_train_res, dinner_test_base = train_ensemble(train_df, dinner_train_features, '석식참여율', test_df)

train_df['lunch_residual'] = train_df['중식참여율'] - lunch_train_res
train_df['dinner_residual'] = train_df['석식참여율'] - dinner_train_res

Training Lunch...
Training Dinner...


In [6]:
# 4. 사후 보정 (Weak Correction)

# (1) 메뉴 보정
LUNCH_SHRINK = 0.35
DINNER_SHRINK = 0.35
RAW_CLIP = 0.03
FINAL_CLIP = 0.04

lunch_menu_cols = [c for c in train_df.columns if c.startswith('kw_')]
dinner_menu_cols = [c for c in train_df.columns if c.startswith('style_')]

lunch_corr = {}
for col in lunch_menu_cols:
    mask = train_df[col] == 1
    lunch_corr[col] = train_df.loc[mask, "lunch_residual"].mean() if mask.sum() >= 5 else 0.0

dinner_corr = {}
for col in dinner_menu_cols:
    mask = train_df[col] == 1
    dinner_corr[col] = train_df.loc[mask, "dinner_residual"].mean() if mask.sum() >= 5 else 0.0

for k in lunch_corr: lunch_corr[k] = np.clip(lunch_corr[k] * LUNCH_SHRINK, -RAW_CLIP, RAW_CLIP)
for k in dinner_corr: dinner_corr[k] = np.clip(dinner_corr[k] * DINNER_SHRINK, -RAW_CLIP, RAW_CLIP)

lunch_menu_adj = np.zeros(len(test_df))
for col in lunch_menu_cols: lunch_menu_adj += test_df[col].values * lunch_corr.get(col, 0.0)
dinner_menu_adj = np.zeros(len(test_df))
for col in dinner_menu_cols: dinner_menu_adj += test_df[col].values * dinner_corr.get(col, 0.0)

lunch_menu_adj = np.clip(lunch_menu_adj, -FINAL_CLIP, FINAL_CLIP)
dinner_menu_adj = np.clip(dinner_menu_adj, -FINAL_CLIP, FINAL_CLIP)

# (2) 날씨 보정 (팀원 모델 가중치)
weather_lunch_signal = (
    0.010 * test_df["is_rain"].values
    - 0.006 * test_df["is_hot"].values
    + 0.004 * test_df["is_cold"].values
)
weather_dinner_signal = (
    0.004 * test_df["is_rain"].values
    + 0.003 * test_df["is_cold"].values
)

# (3) 공휴일 보정 (팀원 모델 가중치)
holiday_lunch_signal = (
    -0.004 * test_df["holiday_before"].values
    + 0.003 * test_df["holiday_after"].values
)
holiday_dinner_signal = (
    -0.005 * test_df["holiday_before"].values
    + 0.003 * test_df["holiday_after"].values
)

# 최종 참여율 계산
final_lunch_ratio = np.clip(lunch_test_base + lunch_menu_adj + weather_lunch_signal + holiday_lunch_signal, 0, 1.5)
final_dinner_ratio = np.clip(dinner_test_base + dinner_menu_adj + weather_dinner_signal + holiday_dinner_signal, 0, 1.5)

# 인원수 환산
pred_lunch = final_lunch_ratio * test_df['식사가능자수']
pred_dinner = final_dinner_ratio * test_df['식사가능자수']

In [7]:
# 5. 저장
sample = pd.read_csv('data/sample_submission.csv')
sample['중식계'] = pred_lunch.values
sample['석식계'] = pred_dinner.values

sample.to_csv('submission/submission_v15.csv', index=False, encoding='utf-8-sig')
print("v15 저장 완료: submission/submission_v15.csv")

v15 저장 완료: submission/submission_v15.csv
